In [ ]:
import pandas as pd
df = pd.read_csv("../data/churn-bigml.csv")

print(df.head())
print(df.shape)
print(df.columns)


  State  Account length  Area code International plan Voice mail plan  \
0    LA             117        408                 No              No   
1    IN              65        415                 No              No   
2    NY             161        415                 No              No   
3    SC             111        415                 No              No   
4    HI              49        510                 No              No   

   Number vmail messages  Total day minutes  Total day calls  \
0                      0              184.5               97   
1                      0              129.1              137   
2                      0              332.9               67   
3                      0              110.4              103   
4                      0              119.3              117   

   Total day charge  Total eve minutes  Total eve calls  Total eve charge  \
0             31.37              351.6               80             29.89   
1             21.95   

In [ ]:
from sklearn.preprocessing import StandardScaler
if 'Churn' in df.columns:
    X = df.drop('Churn', axis=1)
else:
    X = df.copy()
X = pd.get_dummies(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
df = pd.read_csv("../data/churn-bigml.csv")   
print(df.shape)
if 'Churn' in df.columns:
    X = df.drop('Churn', axis=1)
else:
    X = df.copy()
X = pd.get_dummies(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled Shape:", X_scaled.shape)
pca = PCA(n_components=8)   
encoded_features = pca.fit_transform(X_scaled)

print("Encoded Feature Shape:", encoded_features.shape)
kmeans = KMeans(n_clusters=4, random_state=42)
df['Autoencoder_Cluster'] = kmeans.fit_predict(encoded_features)

print(df['Autoencoder_Cluster'].value_counts())


(667, 20)
Scaled Shape: (667, 71)
Encoded Feature Shape: (667, 8)
Autoencoder_Cluster
2    189
0    170
3    160
1    148
Name: count, dtype: int64


In [ ]:
from sklearn.cluster import KMeans
encoded_features = pca.fit_transform(X_scaled)
kmeans = KMeans(n_clusters=4, random_state=42)
df['Autoencoder_Cluster'] = kmeans.fit_predict(encoded_features)
print(df['Autoencoder_Cluster'].value_counts())


Autoencoder_Cluster
1    188
0    180
3    179
2    120
Name: count, dtype: int64


In [12]:
from sklearn.ensemble import RandomForestClassifier

X_features = X
y_clusters = df['Autoencoder_Cluster']

rf = RandomForestClassifier()
rf.fit(X_features, y_clusters)

importances = pd.Series(rf.feature_importances_, index=X_features.columns)
print(importances.sort_values(ascending=False).head(10))


Number vmail messages    0.138210
Voice mail plan_No       0.117376
Voice mail plan_Yes      0.110225
Total eve minutes        0.097873
Total eve charge         0.085188
Total night charge       0.066337
Total night minutes      0.065912
Total day minutes        0.041744
Total day charge         0.040068
Total intl charge        0.039201
dtype: float64


In [17]:
df['Tenure_Group'] = pd.cut(df['Account length'],
                            bins=[0, 50, 100, 200, 500],
                            labels=['New', 'Medium', 'Old', 'Very Old'])

print(df['Tenure_Group'].value_counts())




Tenure_Group
Old         335
Medium      260
New          63
Very Old      9
Name: count, dtype: int64


In [ ]:
df['CLV'] = df['Total day charge'] * df['Account length']

df['CLV_Group'] = pd.qcut(df['CLV'], 4, labels=['Low', 'Medium', 'High', 'Very High'])

print(df[['CLV','CLV_Group']].head())
print(df['CLV_Group'].value_counts())

df['Total_Revenue'] = (df['Total day charge'] + 
                       df['Total eve charge'] + 
                       df['Total night charge'] + 
                       df['Total intl charge'])

df['CLV'] = df['Total_Revenue'] * df['Account length']

df['CLV_Group'] = pd.qcut(df['CLV'], 4, labels=['Low', 'Medium', 'High', 'Very High'])

print(df[['CLV','CLV_Group']].head())



       CLV  CLV_Group
0  3670.29       High
1  1426.75        Low
2  9110.99  Very High
3  2083.47     Medium
4   993.72        Low
CLV_Group
Low          167
Medium       167
Very High    167
High         166
Name: count, dtype: int64
        CLV  CLV_Group
0   8578.44  Very High
1   3523.00        Low
2  14858.69  Very High
3   4556.55     Medium
4   2430.40        Low


In [ ]:
from sklearn.cluster import AgglomerativeClustering
kmeans = KMeans(n_clusters=4)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)
hier = AgglomerativeClustering(n_clusters=3)
df['Hybrid_Cluster'] = hier.fit_predict(X_scaled)
print(df['Hybrid_Cluster'].value_counts())


Hybrid_Cluster
0    453
2    149
1     65
Name: count, dtype: int64
